In [2]:
import numpy as np
import warnings
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.exceptions import ConvergenceWarning

# --- Suppress warnings for cleaner output ---
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# --- 1. Helper Function to Create Dummy Neural Data ---


def create_dummy_data(n_samples=200, n_neurons=100, n_latent_dims=5, noise=1.0):
    """
    Creates a high-dimensional, correlated dataset with a known
    linear and quadratic (synergistic) relationship.

    Y = (Pattern 1) + (Pattern 2) + (Pattern 1 * Pattern 2) + Noise
    """
    print(f"Creating dummy data: {n_samples} samples, {n_neurons} neurons\n")

    # 1. Define the "ground truth" latent patterns
    # These are the 5 "true" patterns hidden in the 100-neuron noise
    latent_space = np.random.randn(n_samples, n_latent_dims)

    # 2. Create the high-dimensional neuron data (X)
    # Project the 5 latent patterns up to 100 neurons
    loadings = np.random.randn(n_latent_dims, n_neurons)
    X = latent_space.dot(loadings)
    X += np.random.randn(n_samples, n_neurons) * 0.5  # Add neuron noise

    # 3. Create the dependent variable (Y) with a quadratic interaction
    # Y depends on the first two latent patterns
    y_linear = latent_space[:, 0] * 2 + latent_space[:, 1] * 1.5

    # This is the "synergy" term you're looking for!
    y_quadratic = latent_space[:, 0] * latent_space[:, 1] * 2.5

    y = 10 + y_linear + y_quadratic + np.random.randn(n_samples) * noise

    return X, y


# --- 2. Function 1: PCR-Quadratic (PCA -> Poly -> Ridge) ---


def fit_pcr_quadratic(X, y, k_outer=5, k_inner=3):
    """
    Fits a low-rank quadratic model using PCA.

    Uses nested cross-validation to tune hyperparameters and evaluate.
    Pipeline: Scale -> PCA -> PolynomialFeatures -> RidgeRegression
    """
    print("--- Starting PCR-Quadratic Model ---")

    # Define the "outer" k-fold loop for evaluation
    outer_cv = KFold(n_splits=k_outer, shuffle=True, random_state=42)
    outer_loop_scores = []

    for i, (train_outer_idx, test_idx) in enumerate(outer_cv.split(X, y)):
        print(f"Outer Fold {i+1}/{k_outer}...")

        # Split into outer-train and test sets
        X_train_outer, y_train_outer = X[train_outer_idx], y[train_outer_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        # --- Define the Model Pipeline ---
        # 1. Scale the data
        # 2. PCA: Reduce n_neurons -> k_components (unsupervised)
        # 3. Poly: Create interactions (Z*Z, Z^2) from k_components
        # 4. Ridge: Fit a regularized model on the polynomial features
        pipeline = Pipeline(
            [
                ("scale", StandardScaler()),
                ("pca", PCA()),
                ("poly", PolynomialFeatures(degree=2, include_bias=False)),
                ("ridge", Ridge(max_iter=10000)),
            ]
        )

        # --- Define the Hyperparameter Grid for the Inner Loop ---
        # We tune PCA's components and Ridge's regularization
        param_grid = {"pca__n_components": [2, 5, 10, 15], "ridge__alpha": [0.1, 1.0, 10.0]}

        # --- Define the "Inner" Loop (GridSearchCV) ---
        # This is your "train" and "validation" split
        # It tunes the hyperparameters on the outer_train set
        inner_cv = KFold(n_splits=k_inner, shuffle=True, random_state=42)

        search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            cv=inner_cv,
            scoring="r2",  # Tune based on R-squared
            n_jobs=-1,  # Use all available cores
        )

        # Run the inner loop (tuning)
        search.fit(X_train_outer, y_train_outer)

        print(f"  Best params: {search.best_params_}")

        # --- Evaluate on the Held-Out Test Set ---
        best_model = search.best_estimator_
        y_pred = best_model.predict(X_test)

        test_r2 = r2_score(y_test, y_pred)
        test_mse = mean_squared_error(y_test, y_pred)

        print(f"  Test R-squared: {test_r2:.4f}")
        outer_loop_scores.append({"r2": test_r2, "mse": test_mse})

    print("--- PCR-Quadratic Finished ---")
    return outer_loop_scores

In [ ]:
X, y = create_dummy_data(n_samples=200, n_neurons=100)

# Run the PCR-Quadratic model
pcr_scores = fit_pcr_quadratic(X, y, k_outer=5, k_inner=3)

In [4]:
def report_scores(scores, name):
    r2_vals = [s["r2"] for s in scores]
    mse_vals = [s["mse"] for s in scores]
    print(f"\nModel: {name}")
    print(f"  Mean R-squared (Test): {np.mean(r2_vals):.4f} " f"± {np.std(r2_vals):.4f}")
    print(f"  Mean MSE (Test):       {np.mean(mse_vals):.4f} " f"± {np.std(mse_vals):.4f}")

In [6]:
report_scores(pcr_scores, "PCR-Quadratic (PCA -> Poly -> Ridge)")


Model: PCR-Quadratic (PCA -> Poly -> Ridge)
  Mean R-squared (Test): 0.9306 ± 0.0216
  Mean MSE (Test):       0.9628 ± 0.1538


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.exceptions import ConvergenceWarning
import warnings

# Suppress warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)


def get_pcr_synergy_report(X, y):
    """
    Finds the best PCR-Quadratic model and returns its
    interpreted coefficients.
    """
    print("Finding best hyperparameters...")

    # --- 1. Define the full pipeline ---
    pipeline = Pipeline(
        [
            ("scale", StandardScaler()),
            ("pca", PCA()),
            ("poly", PolynomialFeatures(degree=2, include_bias=False)),
            ("ridge", Ridge(max_iter=10000)),
        ]
    )

    # --- 2. Define the hyperparameter grid ---
    # We tune PCA's components and Ridge's regularization
    param_grid = {"pca__n_components": [5, 10, 15], "ridge__alpha": [0.1, 1.0, 10.0]}

    # --- 3. Find the best params using GridSearchCV ---
    # We use a simple KFold (not nested) because our goal is
    # to find the best params for a final, interpretable model.
    inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=inner_cv,
        scoring="r2",
        n_jobs=-1,
        refit=True,  # 'refit=True' automatically re-trains the best model on ALL data
    )

    # Run the search
    search.fit(X, y)

    print(f"Best params found: {search.best_params_}")

    # --- 4. Extract Coefficients and Feature Names ---
    # 'best_model' is the pipeline that was re-fit on all of X and y
    best_model = search.best_estimator_

    # Get the coefficient values from the 'ridge' step
    final_coeffs = best_model.named_steps["ridge"].coef_

    # Get the feature names from the 'poly' step
    # This is the magic that maps coefficients to features

    # We need to get the "input features" to PolynomialFeatures
    # which are the PC names (e.g., "PC0", "PC1"...)
    n_pcs = best_model.named_steps["pca"].n_components_
    pc_names = [f"PC{i}" for i in range(n_pcs)]

    # Now get the polynomial feature names
    poly_feature_names = best_model.named_steps["poly"].get_feature_names_out(pc_names)

    # --- 5. Create the Synergy Report ---
    report = pd.Series(final_coeffs, index=poly_feature_names).sort_values(ascending=False)

    return report


# --- Example Usage (assuming you have X and y) ---


In [ ]:
X, y = create_dummy_data(n_samples=200, n_neurons=100)
synergy_report = get_pcr_synergy_report(X, y)

print("\n--- Full Coefficient Report ---")
print(synergy_report)

print("\n--- SYNERGY/INTERACTION REPORT ---")
# Filter for just the interaction terms (those with a space, like 'PC0 PC1')
interaction_report = synergy_report[synergy_report.index.str.contains(" ")]


In [10]:
print(interaction_report)

PC2 PC4    0.053112
PC1 PC3    0.032652
PC0 PC2    0.031016
PC0 PC1    0.017196
PC1 PC4    0.009330
PC0 PC4   -0.003871
PC0 PC3   -0.009601
PC2 PC3   -0.023030
PC3 PC4   -0.029003
PC1 PC2   -0.081428
dtype: float64
